[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/recsys-course/01_data/01_data.ipynb)

# 🗂️ Módulo 01 · Datos de interacción: matrices dispersas, sesgos, splits y baselines

**Curso «Sistemas de Recomendación: de cero a nivel Netflix»** · Bloque I — Fundamentos

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | 3 h (lección) + 2,5 h (proyecto) |
| **GPU** | No necesaria. ≈ 0 unidades de Colab (CPU; MovieLens-1M por defecto, `SCALE="small"` → 100K para iterar) |
| **Prerrequisitos** | Módulo 00 |

> En recomendación, **la mayoría de los errores graves no están en el modelo, sino en los datos y en cómo se parten**. Un split mal hecho puede hacer que un modelo mediocre parezca de estado del arte. Esta lección es la base sobre la que se apoya todo el curso: el pipeline de datos de CineMatch que construyes aquí lo reutilizarán todos los módulos siguientes.

## 🎯 Objetivos de aprendizaje

1. **Representar** interacciones usuario–ítem como matrices dispersas (COO/CSR/CSC) y **estimar** su coste de memoria.
2. **Cuantificar** la *long tail* (curva de Lorenz, Gini, cabeza vs cola) de un dataset.
3. **Explicar y simular** los sesgos principales: popularidad, exposición, selección (MNAR) y posición, y el *feedback loop* que los amplifica.
4. **Construir** señales implícitas (play, % visto, *dwell time*, confianza) a partir de un log de eventos.
5. **Comparar** estrategias de muestreo de negativos y **medir** cuántos "negativos" son en realidad positivos futuros.
6. **Implementar** splits aleatorio, *leave-one-out* y temporal global, y **demostrar** con números el *data leakage* temporal.
7. **Construir y ajustar** baselines fuertes: popularidad global, por segmento y reciente.
8. **Elegir** un dataset público adecuado para cada problema (streaming, e-commerce, noticias, música, CTR, bandits).

### 🔁 Conexión con módulos anteriores
Antes de seguir, intenta responder de memoria (es práctica de recuperación: cuesta un poco y por eso funciona):

1. En el módulo 00, ¿por qué una película no vista **no** es un ejemplo negativo?
2. ¿Qué escalón del embudo de señales (impresión → clic → play → visto ≥ 70 % → like) elegirías como «positivo» para CineMatch, y qué pierdes con esa elección?
3. ¿Qué fila de la tabla «ML clásico vs recomendación» del módulo 00 anticipaba el tema central de este módulo?

<details><summary>Respuestas</summary>
1. Porque mezcla «no le gusta» con «nunca se lo enseñaste / no lo conoce» (sesgo de exposición). Aquí lo medirás: parte de los «negativos» muestreados son positivos futuros (sección 10).
2. No hay respuesta única: «visto ≥ 70 %» es fiable pero escaso y favorece contenido corto; el clic es abundante pero ruidoso. Aquí construirás varias de esas señales (sección 5).
3. «Hay que respetar el tiempo: entrenar con el pasado y evaluar en el futuro». La sección 7 lo convierte en números.
</details>

In [ ]:
!pip install -q pandas pyarrow scipy matplotlib

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp

SEED = 42
rng = np.random.default_rng(SEED)
SCALE = "full"                      # "full" → MovieLens-1M (recomendado) · "small" → MovieLens-100K (solo para iterar)
SIZE = "100k" if SCALE == "small" else "1m"
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COL = {"train": "#4C72B0", "val": "#DD8452", "test": "#C44E52", "gris": "#9E9E9E", "verde": "#55A868"}

In [ ]:
#@title 🔧 Utilidades del curso: cinematch_data.py { display-mode: "form" }
# Escribe los módulos reutilizables del curso en el directorio actual.
# Última versión en GitHub (alternativa):
#   !wget -q https://raw.githubusercontent.com/padillacm/IA-practice/master/recsys-course/01_data/cinematch_data.py
_SOURCES = {}
_SOURCES['cinematch_data.py'] = '"""cinematch_data — pipeline de datos de CineMatch (módulo 01 del curso).\n\nInterfaz estable que reutilizan los módulos posteriores:\n\n    ratings, items = load_movielens("100k")      # o "1m", "latest-small", "25m", "32m"\n    users = load_users("100k")                    # demografía (solo 100k y 1m; si no, None)\n    inter = to_implicit(ratings, threshold=4.0)   # feedback implícito (rating >= 4)\n    inter = filter_k_core(inter, min_user=5, min_item=5)\n    train, val, test = temporal_split(inter, val_frac=0.1, test_frac=0.1)\n    train, test = leave_one_out_split(inter)      # último ítem de cada usuario a test\n    train, test = random_split(inter, test_frac=0.2)\n    X, u2i, i2i = to_csr(train)                   # matriz usuario×ítem scipy.sparse\n    recs = recommend_popular(train, users=test.user_id.unique(), k=10)\n    # recs: dict[user_id -> list[item_id]] ordenado de más a menos recomendado\n\nConvenciones de columnas: user_id (int), item_id (int), rating (float),\ntimestamp (int, segundos Unix). items: item_id, title, year, genres (str "A|B").\n\nSi GroupLens no responde, 100k y 1m se descargan de un espejo público en GitHub\n(mismo contenido); si tampoco hay red, `load_movielens` genera un dataset\nsintético con el mismo esquema (`make_synthetic_movielens`) y avisa.\n"""\nfrom __future__ import annotations\n\nimport io\nimport urllib.request\nimport warnings\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nBASE_URL = "https://files.grouplens.org/datasets/movielens"\nSIZES = {\n    "100k": "ml-100k",\n    "1m": "ml-1m",\n    "latest-small": "ml-latest-small",\n    "25m": "ml-25m",\n    "32m": "ml-32m",\n}\nGENRES_100K = [\n    "unknown", "Action", "Adventure", "Animation", "Children\'s", "Comedy", "Crime",\n    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",\n    "Romance", "Sci-Fi", "Thriller", "War", "Western",\n]\n\n\n# ----------------------------------------------------------------------------\n# Descarga y carga\n# ----------------------------------------------------------------------------\ndef download_movielens(size: str = "100k", data_dir: str | Path = "data") -> Path:\n    """Descarga y descomprime MovieLens desde GroupLens. Devuelve la carpeta."""\n    if size not in SIZES:\n        raise ValueError(f"size debe ser uno de {list(SIZES)}")\n    name = SIZES[size]\n    data_dir = Path(data_dir)\n    folder = data_dir / name\n    if folder.exists():\n        return folder\n    data_dir.mkdir(parents=True, exist_ok=True)\n    url = f"{BASE_URL}/{name}.zip"\n    print(f"Descargando {url} ...")\n    try:\n        with urllib.request.urlopen(url, timeout=60) as resp:\n            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(data_dir)\n    except Exception as e:\n        if size not in MIRRORS:\n            raise\n        print(f"GroupLens no responde ({e.__class__.__name__}); probando espejo público en GitHub…")\n        _download_mirror(size, folder)\n    return folder\n\n\n# Espejos públicos (mismo contenido que GroupLens) por si files.grouplens.org no responde.\nMIRRORS = {\n    "100k": "https://raw.githubusercontent.com/RUCAIBox/RecBole/master/dataset/ml-100k",\n    "1m": "https://raw.githubusercontent.com/khanhnamle1994/movielens/master",\n}\n\n\ndef _write_dat(df: pd.DataFrame, path: Path) -> None:\n    """Escribe en el formato \'::\' de MovieLens-1M."""\n    lines = ("::".join(map(str, row)) for row in df.itertuples(index=False))\n    path.write_text("\\n".join(lines) + "\\n", encoding="latin-1", errors="replace")\n\n\ndef _download_mirror(size: str, folder: Path) -> None:\n    """Descarga el espejo y lo reescribe con los nombres y el formato originales de\n    GroupLens, para que `_read_folder` y `load_users` funcionen sin cambios."""\n    import csv\n    base, tmp = MIRRORS[size], folder.with_name(folder.name + ".tmp")\n    tmp.mkdir(parents=True, exist_ok=True)\n    if size == "100k":\n        rd = lambda f: pd.read_csv(f"{base}/ml-100k.{f}", sep="\\t", quoting=csv.QUOTE_NONE)\n        inter, it, us = rd("inter"), rd("item"), rd("user")\n        inter.columns, it.columns = ["u", "i", "r", "t"], ["item_id", "title", "year", "genres"]\n        inter.astype("int64").to_csv(tmp / "u.data", sep="\\t", header=False, index=False)\n        year = pd.to_numeric(it["year"], errors="coerce")\n        title = [f"{t} ({int(y)})" if y == y else t for t, y in zip(it["title"], year)]\n        flags = pd.DataFrame({g: it["genres"].str.split(" ").apply(lambda gs: int(g in gs))\n                              for g in GENRES_100K})\n        u_item = pd.concat([it[["item_id"]], pd.Series(title, name="title"),\n                            pd.DataFrame({"rd": "", "vrd": "", "url": ""}, index=it.index), flags], axis=1)\n        u_item.to_csv(tmp / "u.item", sep="|", header=False, index=False, encoding="latin-1",\n                      errors="replace", quoting=csv.QUOTE_NONE, escapechar="\\\\")\n        us.to_csv(tmp / "u.user", sep="|", header=False, index=False)\n    else:  # 1m\n        rd = lambda f: pd.read_csv(f"{base}/{f}.csv", sep="\\t", index_col=0, encoding="latin-1")\n        _write_dat(rd("ratings")[["user_id", "movie_id", "rating", "timestamp"]], tmp / "ratings.dat")\n        _write_dat(rd("movies")[["movie_id", "title", "genres"]], tmp / "movies.dat")\n        _write_dat(rd("users")[["user_id", "gender", "age", "occupation", "zipcode"]], tmp / "users.dat")\n    tmp.rename(folder)\n\n\ndef _split_title_year(titles: pd.Series) -> tuple[pd.Series, pd.Series]:\n    year = titles.str.extract(r"\\((\\d{4})\\)\\s*$")[0].astype("float")\n    clean = titles.str.replace(r"\\s*\\(\\d{4}\\)\\s*$", "", regex=True)\n    return clean, year\n\n\ndef _read_folder(size: str, folder: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    if size == "100k":\n        ratings = pd.read_csv(folder / "u.data", sep="\\t",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        cols = ["item_id", "title", "release_date", "video_release_date", "url"] + GENRES_100K\n        raw = pd.read_csv(folder / "u.item", sep="|", names=cols, encoding="latin-1")\n        flags = raw[GENRES_100K].values.astype(bool)\n        genres = ["|".join(g for g, f in zip(GENRES_100K, row) if f) or "unknown" for row in flags]\n        items = pd.DataFrame({"item_id": raw["item_id"], "title": raw["title"].fillna(""),\n                              "genres": genres})\n    elif size == "1m":\n        ratings = pd.read_csv(folder / "ratings.dat", sep="::", engine="python",\n                              names=["user_id", "item_id", "rating", "timestamp"])\n        items = pd.read_csv(folder / "movies.dat", sep="::", engine="python",\n                            names=["item_id", "title", "genres"], encoding="latin-1")\n    else:\n        ratings = pd.read_csv(folder / "ratings.csv").rename(\n            columns={"userId": "user_id", "movieId": "item_id"})\n        items = pd.read_csv(folder / "movies.csv").rename(columns={"movieId": "item_id"})\n    items["title"], items["year"] = _split_title_year(items["title"].astype(str))\n    return ratings, items[["item_id", "title", "year", "genres"]]\n\n\ndef load_movielens(size: str = "100k", data_dir: str | Path = "data",\n                   synthetic_fallback: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Devuelve (ratings, items) con el esquema estándar de CineMatch.\n\n    ratings: user_id, item_id, rating (float32), timestamp (int64), ordenado por tiempo.\n    items:   item_id, title, year, genres ("Action|Comedy").\n    """\n    try:\n        folder = download_movielens(size, data_dir)\n        ratings, items = _read_folder(size, folder)\n    except Exception as e:  # sin red, URL caída, etc.\n        if not synthetic_fallback:\n            raise\n        warnings.warn(f"No se pudo descargar MovieLens-{size} ({e!r}). "\n                      "Usando datos SINTÉTICOS con el mismo esquema.")\n        ratings, items = make_synthetic_movielens()\n    ratings = ratings.astype({"user_id": "int64", "item_id": "int64",\n                              "rating": "float32", "timestamp": "int64"})\n    ratings = ratings.sort_values(["timestamp", "user_id", "item_id"], kind="stable")\n    return ratings.reset_index(drop=True), items.reset_index(drop=True)\n\n\ndef load_users(size: str = "100k", data_dir: str | Path = "data") -> pd.DataFrame | None:\n    """Demografía de usuarios (user_id, age, gender, occupation). Solo 100k y 1m."""\n    try:\n        folder = download_movielens(size, data_dir)\n    except Exception:\n        return None\n    if size == "100k":\n        return pd.read_csv(folder / "u.user", sep="|",\n                           names=["user_id", "age", "gender", "occupation", "zip"])\n    if size == "1m":\n        return pd.read_csv(folder / "users.dat", sep="::", engine="python",\n                           names=["user_id", "gender", "age", "occupation", "zip"])\n    return None\n\n\ndef make_synthetic_movielens(n_users: int = 943, n_items: int = 1682,\n                             n_ratings: int = 100_000, seed: int = 42\n                             ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Dataset sintético con forma de MovieLens: long tail, actividad sesgada,\n    gustos latentes por género y deriva temporal (los ítems nuevos van llegando)."""\n    rng = np.random.default_rng(seed)\n    genres = GENRES_100K[1:]\n    t0, t1 = 874_724_710, 893_286_638  # mismo rango temporal que ML-100K\n    # Ítems: género principal, popularidad intrínseca (Zipf) y fecha de llegada\n    item_genre = rng.integers(0, len(genres), n_items)\n    item_pop = 1.0 / np.arange(1, n_items + 1) ** 1.1\n    rng.shuffle(item_pop)\n    item_arrival = t0 + (t1 - t0) * np.where(rng.random(n_items) < 0.5, 0.0,\n                                             rng.uniform(0, 0.95, n_items))  # catálogo inicial + estrenos\n    # Usuarios: actividad log-normal, afinidad por géneros y fecha de alta\n    activity = rng.lognormal(0, 1.0, n_users)\n    activity = np.maximum(20, activity / activity.sum() * n_ratings).astype(int)\n    taste = rng.dirichlet(np.full(len(genres), 0.3), n_users)\n    user_start = t0 + (t1 - t0) * rng.uniform(0, 0.9, n_users)\n    item_quality = rng.normal(0, 0.45, n_items)   # unas películas son mejores que otras\n    user_bias = rng.normal(0, 0.35, n_users)      # hay usuarios generosos y exigentes\n    rows = []\n    for u in range(n_users):\n        start = user_start[u]\n        end = min(t1, start + rng.uniform(1, 120) * 86_400)\n        avail = item_arrival <= end\n        n = int(min(activity[u], avail.sum() // 2))\n        ts = np.sort(rng.uniform(start, end, n))\n        age_days = np.maximum(0, end - item_arrival) / 86_400\n        hype = np.where(item_arrival > t0, 1 + 30 * np.exp(-age_days / 30), 1.0)  # los estrenos tienen su momento\n        w = item_pop * hype * (0.15 + taste[u][item_genre]) * avail\n        chosen = rng.choice(n_items, size=n, replace=False, p=w / w.sum())\n        ts = np.maximum(ts, item_arrival[chosen] + 3600)\n        aff = taste[u][item_genre[chosen]] * len(genres)\n        r = np.clip(np.round(3.2 + 0.6 * np.log1p(aff) + item_quality[chosen] + user_bias[u]\n                             + rng.normal(0, 0.7, n)), 1, 5)\n        rows.append(pd.DataFrame({"user_id": u + 1, "item_id": chosen + 1,\n                                  "rating": r, "timestamp": ts.astype(np.int64)}))\n    ratings = pd.concat(rows, ignore_index=True)\n    second = rng.integers(0, len(genres), n_items)\n    items = pd.DataFrame({\n        "item_id": np.arange(1, n_items + 1),\n        "title": [f"Película sintética {i}" for i in range(1, n_items + 1)],\n        "year": (1930 + 66 * rng.beta(4, 1.5, n_items)).round(),\n        "genres": [genres[a] if a == b else f"{genres[a]}|{genres[b]}"\n                   for a, b in zip(item_genre, second)],\n    })\n    return ratings, items\n\n\n# ----------------------------------------------------------------------------\n# Limpieza y señales\n# ----------------------------------------------------------------------------\ndef to_implicit(ratings: pd.DataFrame, threshold: float = 4.0) -> pd.DataFrame:\n    """Convierte ratings explícitos en interacciones implícitas positivas\n    (rating >= threshold). Mantiene la columna rating por si se quiere ponderar."""\n    out = ratings[ratings["rating"] >= threshold].copy()\n    return out.reset_index(drop=True)\n\n\ndef dedup_interactions(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:\n    """Elimina pares (usuario, ítem) duplicados quedándose con el primero/último en el tiempo."""\n    return (df.sort_values("timestamp", kind="stable")\n              .drop_duplicates(["user_id", "item_id"], keep=keep)\n              .reset_index(drop=True))\n\n\ndef filter_k_core(df: pd.DataFrame, min_user: int = 5, min_item: int = 5,\n                  max_iter: int = 100) -> pd.DataFrame:\n    """k-core iterativo: repite hasta que todo usuario tenga >= min_user\n    interacciones y todo ítem >= min_item (filtrar uno puede romper el otro)."""\n    for _ in range(max_iter):\n        n0 = len(df)\n        ic = df["item_id"].map(df["item_id"].value_counts())\n        df = df[ic >= min_item]\n        uc = df["user_id"].map(df["user_id"].value_counts())\n        df = df[uc >= min_user]\n        if len(df) == n0:\n            break\n    return df.reset_index(drop=True)\n\n\n# ----------------------------------------------------------------------------\n# Splits\n# ----------------------------------------------------------------------------\ndef _drop_cold(train: pd.DataFrame, other: pd.DataFrame, users: bool, items: bool) -> pd.DataFrame:\n    if users:\n        other = other[other["user_id"].isin(train["user_id"].unique())]\n    if items:\n        other = other[other["item_id"].isin(train["item_id"].unique())]\n    return other.reset_index(drop=True)\n\n\ndef random_split(df: pd.DataFrame, test_frac: float = 0.2, seed: int = 42,\n                 drop_cold: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Split aleatorio de interacciones (¡filtra el futuro al train! solo como contraejemplo)."""\n    mask = np.random.default_rng(seed).random(len(df)) < test_frac\n    train, test = df[~mask].reset_index(drop=True), df[mask]\n    return train, _drop_cold(train, test, drop_cold, drop_cold)\n\n\ndef leave_one_out_split(df: pd.DataFrame, n_test: int = 1, min_train: int = 1\n                        ) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Las últimas n_test interacciones (en el tiempo) de cada usuario van a test.\n    Ojo: no hay un corte temporal global → el train de un usuario puede contener\n    interacciones posteriores al test de otro (leakage temporal entre usuarios)."""\n    df = df.sort_values(["user_id", "timestamp"], kind="stable")\n    rank_from_end = df.groupby("user_id").cumcount(ascending=False)\n    n_user = df.groupby("user_id")["item_id"].transform("size")\n    is_test = (rank_from_end < n_test) & (n_user >= n_test + min_train)\n    return df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)\n\n\ndef temporal_split(df: pd.DataFrame, val_frac: float = 0.1, test_frac: float = 0.1,\n                   drop_cold_users: bool = True, drop_cold_items: bool = True\n                   ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Split temporal GLOBAL: train = pasado, val = siguiente val_frac del tiempo\n    (por cuantiles de eventos), test = último test_frac. Es lo más parecido a\n    producción: entrenas con todo lo anterior a T y sirves después de T.\n    Devuelve (train, val, test); val puede estar vacío si val_frac == 0."""\n    ts = df["timestamp"]\n    t_test = ts.quantile(1 - test_frac)\n    t_val = ts.quantile(1 - test_frac - val_frac) if val_frac > 0 else t_test\n    train = df[ts < t_val].reset_index(drop=True)\n    val = df[(ts >= t_val) & (ts < t_test)]\n    test = df[ts >= t_test]\n    val = _drop_cold(train, val, drop_cold_users, drop_cold_items)\n    # Para test, "conocido" = visto en train o val (en producción reentrenas antes de servir)\n    seen = pd.concat([train, val])\n    test = _drop_cold(seen, test, drop_cold_users, drop_cold_items)\n    return train, val, test\n\n\n# ----------------------------------------------------------------------------\n# Matrices dispersas\n# ----------------------------------------------------------------------------\ndef build_mappings(df: pd.DataFrame) -> tuple[dict, dict]:\n    """Mapeos id original -> índice contiguo 0..n-1 (usuarios e ítems)."""\n    users = np.sort(df["user_id"].unique())\n    items = np.sort(df["item_id"].unique())\n    return ({u: i for i, u in enumerate(users)}, {it: j for j, it in enumerate(items)})\n\n\ndef to_csr(df: pd.DataFrame, user2idx: dict | None = None, item2idx: dict | None = None,\n           value_col: str | None = None) -> tuple[sp.csr_matrix, dict, dict]:\n    """Matriz usuario×ítem CSR. value_col=None → 1.0 por interacción (implícito).\n    Las filas/ítems fuera del mapeo se descartan. Devuelve (X, user2idx, item2idx)."""\n    if user2idx is None or item2idx is None:\n        user2idx, item2idx = build_mappings(df)\n    r = df["user_id"].map(user2idx)\n    c = df["item_id"].map(item2idx)\n    ok = r.notna() & c.notna()\n    vals = np.ones(ok.sum(), dtype=np.float32) if value_col is None \\\n        else df.loc[ok, value_col].to_numpy(np.float32)\n    X = sp.csr_matrix((vals, (r[ok].astype(int), c[ok].astype(int))),\n                      shape=(len(user2idx), len(item2idx)))\n    X.sum_duplicates()\n    return X, user2idx, item2idx\n\n\n# ----------------------------------------------------------------------------\n# Baselines\n# ----------------------------------------------------------------------------\ndef _seen_by_user(train: pd.DataFrame) -> dict:\n    return train.groupby("user_id")["item_id"].agg(set).to_dict()\n\n\ndef _topk_excluding(ranked: np.ndarray, seen: set, k: int) -> list:\n    out = []\n    for it in ranked:\n        if it not in seen:\n            out.append(int(it))\n            if len(out) == k:\n                break\n    return out\n\n\ndef popularity_ranking(train: pd.DataFrame, window_days: float | None = None,\n                       half_life_days: float | None = None) -> np.ndarray:\n    """Ítems ordenados por popularidad. window_days: solo cuenta los últimos N días.\n    half_life_days: pondera cada evento por 0.5**(edad/half_life) (recencia suave)."""\n    df = train\n    t_max = df["timestamp"].max()\n    if window_days is not None:\n        df = df[df["timestamp"] >= t_max - window_days * 86_400]\n    if half_life_days is not None:\n        age = (t_max - df["timestamp"]) / 86_400\n        w = 0.5 ** (age / half_life_days)\n        scores = w.groupby(df["item_id"]).sum()\n    else:\n        scores = df["item_id"].value_counts()\n    return scores.sort_values(ascending=False, kind="stable").index.to_numpy()\n\n\ndef recommend_popular(train: pd.DataFrame, users, k: int = 10, exclude_seen: bool = True,\n                      window_days: float | None = None,\n                      half_life_days: float | None = None) -> dict:\n    """Top-K más populares (opcionalmente recientes) para cada usuario de `users`."""\n    ranked = popularity_ranking(train, window_days, half_life_days)\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranked, seen.get(u, set()), k) for u in users}\n\n\ndef recommend_popular_by_segment(train: pd.DataFrame, users, segment_of: dict, k: int = 10,\n                                 exclude_seen: bool = True, min_segment_events: int = 50) -> dict:\n    """Popularidad dentro del segmento del usuario (p. ej. edad, país, género favorito).\n    `segment_of`: dict user_id -> segmento. Segmentos con pocos eventos → popularidad global."""\n    seg = train["user_id"].map(segment_of)\n    global_rank = popularity_ranking(train)\n    ranks = {}\n    for s, g in train.groupby(seg):\n        if len(g) >= min_segment_events:\n            r = g["item_id"].value_counts().index.to_numpy()\n            # completar con el ranking global para no quedarnos cortos\n            ranks[s] = np.concatenate([r, global_rank[~np.isin(global_rank, r)]])\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(ranks.get(segment_of.get(u), global_rank), seen.get(u, set()), k)\n            for u in users}\n\n\ndef recommend_random(train: pd.DataFrame, users, k: int = 10, seed: int = 42,\n                     exclude_seen: bool = True) -> dict:\n    """Recomendador aleatorio: el suelo absoluto de cualquier comparación."""\n    rng = np.random.default_rng(seed)\n    items = train["item_id"].unique()\n    seen = _seen_by_user(train) if exclude_seen else {}\n    return {u: _topk_excluding(rng.permutation(items), seen.get(u, set()), k) for u in users}\n\n\n# ----------------------------------------------------------------------------\n# Persistencia\n# ----------------------------------------------------------------------------\ndef save_splits(out_dir: str | Path, **frames: pd.DataFrame) -> Path:\n    """Guarda cada DataFrame como parquet: save_splits("data/cinematch", train=..., test=...)."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    for name, f in frames.items():\n        f.to_parquet(out_dir / f"{name}.parquet", index=False)\n    return out_dir\n\n\ndef load_splits(out_dir: str | Path, names=("train", "val", "test", "items")) -> dict:\n    out_dir = Path(out_dir)\n    return {n: pd.read_parquet(out_dir / f"{n}.parquet") for n in names\n            if (out_dir / f"{n}.parquet").exists()}\n\n\n# ----------------------------------------------------------------------------\n# Pipeline completo de CineMatch (una llamada)\n# ----------------------------------------------------------------------------\ndef prepare_cinematch(size: str = "100k", threshold: float | None = 4.0, min_user: int = 5,\n                      min_item: int = 5, val_frac: float = 0.1, test_frac: float = 0.1,\n                      data_dir: str | Path = "data", out_dir: str | Path | None = None) -> dict:\n    """Pipeline estándar del curso:\n    carga → dedup → implícito (rating >= threshold; None = no filtrar) → split temporal\n    global → k-core SOLO sobre train (no miramos el futuro para filtrar) → val/test\n    restringidos a usuarios e ítems conocidos en train.\n\n    Devuelve {"train", "val", "test", "items", "ratings"} (DataFrames) y, si\n    out_dir no es None, los guarda en parquet."""\n    ratings, items = load_movielens(size, data_dir)\n    inter = dedup_interactions(ratings)\n    if threshold is not None:\n        inter = to_implicit(inter, threshold)\n    train, val, test = temporal_split(inter, val_frac, test_frac,\n                                      drop_cold_users=False, drop_cold_items=False)\n    train = filter_k_core(train, min_user, min_item)\n    val = _drop_cold(train, val, True, True)\n    test = _drop_cold(train, test, True, True)\n    out = {"train": train, "val": val, "test": test, "items": items, "ratings": ratings}\n    if out_dir is not None:\n        save_splits(out_dir, train=train, val=val, test=test, items=items)\n    return out\n'
for _name, _src in _SOURCES.items():
    with open(_name, 'w', encoding='utf-8') as _fh:
        _fh.write(_src)
    print('✔ escrito', _name)

In [ ]:
import cinematch_data as cd

ratings, items = cd.load_movielens(SIZE)
users = cd.load_users(SIZE)             # demografía (None si no hay red / dataset sin usuarios)
print(ratings.shape, items.shape, None if users is None else users.shape)
ratings.head()

---
## 1. 💡 Intuición: de un log de eventos a una matriz

Una plataforma no "tiene ratings": tiene un **log de eventos** — una tabla larguísima, ordenada en el tiempo, con filas del tipo `(usuario, ítem, tipo de evento, timestamp, contexto)`. Todo recomendador empieza **agregando** ese log en una matriz usuario × ítem $\mathbf{R} \in \mathbb{R}^{|U| \times |I|}$ y decidiendo:

- **Qué evento cuenta como señal** (¿rating ≥ 4? ¿play? ¿≥ 70 % visto?).
- **Qué valor** poner en la celda (1, el rating, una confianza).
- **Qué pasado** usar para entrenar y **qué futuro** para evaluar.

Las tres decisiones cambian el resultado más que la elección de modelo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.1, 1]})
log = pd.DataFrame({"usuario": ["Ana", "Ana", "Bruno", "Carla", "Ana", "Bruno", "Carla", "Bruno"],
                    "ítem": ["Matrix", "Alien", "Matrix", "Titanic", "Interstellar", "Alien", "Matrix", "Titanic"],
                    "evento": ["play 95%", "play 12%", "play 100%", "rating 5", "play 80%", "play 60%", "impresión", "impresión"],
                    "t": [1, 2, 3, 4, 5, 6, 7, 8]})
axes[0].axis("off")
tabla = axes[0].table(cellText=log.values, colLabels=log.columns, loc="center", cellLoc="center")
tabla.scale(1, 1.4)
axes[0].set_title("Log de eventos (ordenado en el tiempo)")
U, I = ["Ana", "Bruno", "Carla"], ["Matrix", "Alien", "Titanic", "Interstellar"]
M_ = np.full((3, 4), np.nan)
for _, r in log.iterrows():
    if r.evento.startswith("play"):
        M_[U.index(r.usuario), I.index(r["ítem"])] = int(r.evento.split()[1][:-1]) / 100
    elif r.evento.startswith("rating"):
        M_[U.index(r.usuario), I.index(r["ítem"])] = 1.0
axes[1].imshow(np.nan_to_num(M_, nan=-0.3), cmap="Blues", vmin=-0.3, vmax=1)
for a in range(3):
    for b in range(4):
        axes[1].text(b, a, "·" if np.isnan(M_[a, b]) else f"{M_[a, b]:.2f}", ha="center", va="center")
axes[1].set_xticks(range(4), I); axes[1].set_yticks(range(3), U)
axes[1].set_title("→ agregado: matriz usuario × ítem (valor = fracción vista)")
plt.tight_layout(); plt.show()

Fíjate en dos detalles que parecen menores y no lo son:
- Las **impresiones sin play** (Carla–Matrix, Bruno–Titanic) son la mejor fuente de **negativos reales**: se lo enseñaste y no lo quiso. Si no las logueas, no las tendrás nunca. Casi ningún dataset público las trae (MIND, KuaiRand y Open Bandit sí).
- Ana vio *Alien* al 12 %: ¿es positivo o negativo? Definir el umbral es una decisión de producto.

---
## 2. 📐 Matrices dispersas

Con $|U|$ usuarios y $|I|$ ítems, la matriz densa ocupa $|U|\cdot|I|$ celdas, pero solo hay $\text{nnz}$ (*number of non-zeros*) observadas. La **densidad** $\rho = \text{nnz} / (|U|\cdot|I|)$ suele estar entre $10^{-2}$ y $10^{-6}$.

Formatos de `scipy.sparse` que usarás constantemente:
- **COO** (*coordinate*): tres arrays `row`, `col`, `data`. Ideal para **construir**.
- **CSR** (*compressed sparse row*): `indptr` (longitud $|U|+1$), `indices`, `data`. La fila $u$ son las posiciones `indptr[u]:indptr[u+1]`. Acceso rápido **por usuario** (su historial) y productos matriz-vector.
- **CSC**: lo mismo por columnas → acceso rápido **por ítem**.

Memoria (float32 + índices int32): denso $= 4\,|U||I|$ bytes; CSR $\approx 8\,\text{nnz} + 4(|U|+1)$ bytes.

In [ ]:
# CSR "a mano" para entender la estructura
filas = np.array([0, 0, 1, 1, 1, 2])        # usuario
cols = np.array([0, 3, 0, 1, 2, 2])         # ítem
vals = np.array([1, 1, 1, 1, 1, 1], dtype=np.float32)
X_toy = sp.csr_matrix((vals, (filas, cols)), shape=(3, 4))
print("Densa:\n", X_toy.toarray())
print("indptr :", X_toy.indptr, " ← la fila u ocupa indices[indptr[u]:indptr[u+1]]")
print("indices:", X_toy.indices)
print("data   :", X_toy.data)
print("Historial del usuario 1:", X_toy.indices[X_toy.indptr[1]:X_toy.indptr[2]])

In [ ]:
X, u2i, i2i = cd.to_csr(ratings, value_col="rating")
print(f"MovieLens-{SIZE}: {X.shape}, nnz = {X.nnz:,}, densidad = {X.nnz / np.prod(X.shape):.3%}")

# Memoria de cada formato para datasets reales (cifras oficiales de cada dataset)
datasets = {"ML-100K": (943, 1_682, 100_000), "ML-1M": (6_040, 3_706, 1_000_209),
            "ML-25M": (162_541, 59_047, 25_000_095), "ML-32M": (200_948, 87_585, 32_000_204),
            "Netflix Prize": (480_189, 17_770, 100_480_507)}
mem = pd.DataFrame({n: {"densidad": nnz / (u * i), "denso_GB": 4 * u * i / 1e9,
                        "CSR_GB": (8 * nnz + 4 * (u + 1)) / 1e9} for n, (u, i, nnz) in datasets.items()}).T
fig, ax = plt.subplots(figsize=(9, 3.6))
mem[["denso_GB", "CSR_GB"]].plot.bar(ax=ax, logy=True, color=[COL["test"], COL["train"]], rot=0)
ax.set(ylabel="GB (escala log)", title="Memoria: matriz densa vs CSR (float32)")
for k, d in enumerate(mem.densidad):
    ax.text(k, mem.denso_GB.iloc[k] * 1.3, f"ρ={d:.2%}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
mem.round(4)

La matriz densa del Netflix Prize ocuparía ~34 GB; en CSR, menos de 1 GB. Y una operación clave en recomendación — la **co-ocurrencia ítem-ítem** $\mathbf{X}^\top\mathbf{X}$ ("cuántos usuarios vieron $i$ y $j$") — es un simple producto disperso:

In [ ]:
Xb = (X > 0).astype(np.float32)          # binaria
t0 = time.perf_counter()
co = (Xb.T @ Xb).tocsr()                 # ítem × ítem
print(f"Co-ocurrencia {co.shape} en {time.perf_counter() - t0:.3f} s · nnz = {co.nnz:,}")
idx2item = {j: it for it, j in i2i.items()}
top_item = int(np.asarray(Xb.sum(0)).ravel().argmax())
fila = co[top_item].toarray().ravel(); fila[top_item] = 0
vecinos = [idx2item[j] for j in np.argsort(-fila)[:5]]
print("Más co-vistas con", items.set_index("item_id").title.get(idx2item[top_item]), "→",
      items.set_index("item_id").loc[vecinos, "title"].tolist())

---
## 3. Long tail: cabeza y cola

La popularidad de los ítems sigue aproximadamente una **ley de potencias**: unos pocos ítems (la **cabeza**, *short head*) concentran la mayoría de interacciones y miles (la **cola**, *long tail*) tienen muy pocas. Dos formas de medirlo:

- **Curva de Lorenz**: % acumulado de interacciones frente a % de ítems (de menos a más popular).
- **Índice de Gini**: $G = \frac{\sum_{k=1}^{n}(2k-n-1)\,x_{(k)}}{n\sum_k x_{(k)}}$ con $x_{(k)}$ ordenadas ascendentemente. $G = 0$: todos igual de populares; $G \to 1$: todo en un ítem.

¿Por qué importa? (1) Los modelos aprenden mucho mejor la cabeza (más datos) → tienden a recomendar más de lo mismo. (2) La cola es donde está el **descubrimiento** y el valor para productores de nicho. (3) Las métricas promediadas sobre interacciones las domina la cabeza.

In [ ]:
def gini(x) -> float:
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))

pop = ratings.item_id.value_counts()
cum = pop.cumsum() / pop.sum()
n_head = int((cum <= 0.5).sum()) + 1                      # ítems que acumulan el 50 % de interacciones
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
rank = np.arange(1, len(pop) + 1)
axes[0].loglog(rank, pop.values, ".", ms=3, color=COL["train"])
axes[0].axvline(n_head, color=COL["test"], ls="--")
axes[0].text(n_head * 1.15, pop.max() / 2, f"cabeza: {n_head} ítems\n({n_head / len(pop):.0%}) = 50 % de eventos",
             color=COL["test"], fontsize=9)
axes[0].set(title="Popularidad por rango (log-log)", xlabel="rango del ítem", ylabel="nº interacciones")
x = np.sort(pop.values)
axes[1].plot(np.linspace(0, 1, len(x)), np.cumsum(x) / x.sum(), color=COL["val"], label=f"Lorenz (Gini={gini(x):.2f})")
axes[1].plot([0, 1], [0, 1], "--", color=COL["gris"], label="igualdad")
axes[1].set(title="Curva de Lorenz de la popularidad", xlabel="fracción de ítems", ylabel="fracción de interacciones")
axes[1].legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** en el panel izquierdo, los puntos caen aproximadamente en una **recta en escala log-log** (ley de potencias): el ítem del rango 10 tiene muchas más interacciones que el del rango 1.000. La línea roja marca qué pocos ítems acumulan la mitad de los eventos. En el derecho, cuanto más se separa la curva de la diagonal, mayor el Gini. Guarda este número: en el módulo 02 medirás el Gini **de las recomendaciones** (no del dataset) y verás que casi todos los modelos lo **aumentan**.

---
## 4. ⚠️ Los sesgos de los datos de recomendación

Tus datos **no son una muestra aleatoria** de las preferencias de los usuarios. Son el resultado de lo que tu sistema enseñó, de lo que la gente eligió mirar y de dónde lo puso. Los cuatro sesgos clásicos (ver la revisión de Chen et al., 2023, *Bias and Debias in Recommender System*):

| Sesgo | Qué pasa | Consecuencia |
|---|---|---|
| **Selección / MNAR** (*missing not at random*) | Los usuarios valoran lo que eligen ver, y eligen lo que creen que les gustará | Los ratings observados están inflados; "no observado" ≠ "valor medio" |
| **Exposición** | Solo puedes interactuar con lo que se te mostró | "No interactuó" mezcla "no le gusta" con "no lo vio" |
| **Posición** | Lo de arriba recibe más clics *porque está arriba* | El clic mezcla relevancia y posición (módulo 07) |
| **Popularidad** | El modelo sobre-recomienda la cabeza | Menos diversidad; se autorrefuerza (*feedback loop*) |

### 🧪 Simulación 1 — MNAR: lo que observas no es lo que hay
Generamos las preferencias "verdaderas" de todos los usuarios para todas las películas y hacemos que la probabilidad de **observar** un rating crezca con el propio rating (la gente ve y valora lo que le gusta). Marlin & Zemel (2009) lo midieron con usuarios reales de Yahoo! Music pidiéndoles valorar canciones **al azar**: la distribución era muy distinta de la de las valoraciones voluntarias.

In [ ]:
n_u, n_i, d = 2000, 500, 8
P, Q = rng.normal(size=(n_u, d)), rng.normal(size=(n_i, d))
verdad = np.clip(np.round(3 + 0.55 * (P @ Q.T) / np.sqrt(d) * 2 + rng.normal(0, 0.5, (n_u, n_i))), 1, 5)
p_obs = np.array([0.005, 0.01, 0.03, 0.08, 0.15])[verdad.astype(int) - 1]   # P(observar | rating)
observado = verdad[rng.random(verdad.shape) < p_obs]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
b = np.arange(1, 6)
ax.bar(b - 0.2, [np.mean(verdad == k) for k in b], 0.4, label="todas las preferencias (verdad)", color=COL["gris"])
ax.bar(b + 0.2, [np.mean(observado == k) for k in b], 0.4, label="ratings observados", color=COL["train"])
ax.set(xlabel="rating", ylabel="proporción", title=f"MNAR: media real {verdad.mean():.2f} vs observada {observado.mean():.2f}")
ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** la distribución observada está desplazada hacia 4–5★ y la media observada es mayor que la real. Un modelo entrenado solo con lo observado «cree» que las películas son mejores de lo que son; y su RMSE en test (también observado) **no lo detecta**, porque test tiene el mismo sesgo.

### 🧪 Simulación 2 — Sesgo de posición
Ordenamos ítems **al azar** (la relevancia no depende de la posición) y simulamos clics con un modelo de examen: el usuario mira la posición $k$ con probabilidad $P(E=1\mid k) = 1/k^{\eta}$ y clica si la mira y le interesa — el *position-based model* de los trabajos de LTR insesgado (Joachims et al., 2017).

In [ ]:
K, n_sesiones, eta = 10, 50_000, 1.0
relevante = rng.random((n_sesiones, K)) < 0.15            # relevancia independiente de la posición
examina = rng.random((n_sesiones, K)) < 1 / np.arange(1, K + 1) ** eta
clic = relevante & examina
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(range(1, K + 1), clic.mean(0), color=COL["val"], label="CTR observado")
ax.plot(range(1, K + 1), relevante.mean(0), "k--", label="relevancia real (constante)")
ax.set(xlabel="posición en la lista", ylabel="tasa", title="Sesgo de posición: mismo contenido, CTR muy distinto")
ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar:** la línea discontinua es plana (todas las posiciones son igual de relevantes) pero las barras caen como $1/k$. Si entrenas un modelo de clic con estos logs **sin** decirle la posición, aprenderá que «lo que estaba arriba es mejor» y reforzará el orden antiguo. El módulo 07 corrige esto (IPS, PAL) y el 14 lo generaliza (propensiones).

### 🧪 Simulación 3 — El *feedback loop* de la popularidad
Un catálogo de 300 ítems con un "atractivo" real casi plano. Cada ronda recomendamos a cada usuario los 10 ítems con más clics acumulados (más un 2 % de exploración aleatoria), los usuarios clican según el atractivo real, y reentrenamos. Mira cómo se dispara la concentración aunque los ítems sean casi iguales.

In [ ]:
def simular_bucle(n_items=300, n_users=500, rondas=30, k=10, explora=0.02, seed=SEED):
    r = np.random.default_rng(seed)
    atractivo = r.uniform(0.04, 0.06, n_items)            # ¡casi todos iguales!
    clics = np.ones(n_items)
    ginis = []
    for _ in range(rondas):
        top = np.argsort(-clics)[:k]
        for _u in range(n_users):
            lista = top.copy()
            mask = r.random(k) < explora
            lista[mask] = r.integers(0, n_items, mask.sum())
            clics[lista] += r.random(k) < atractivo[lista]
        ginis.append(gini(clics))
    return np.array(ginis), clics, atractivo

ginis, clics_fin, atractivo = simular_bucle()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(ginis, marker="o", color=COL["test"])
axes[0].set(title="Gini de los clics acumulados por ronda", xlabel="ronda de reentreno", ylabel="Gini")
axes[1].scatter(atractivo, clics_fin, s=8, color=COL["train"])
axes[1].set(title="Atractivo real vs clics al final", xlabel="atractivo real", ylabel="clics acumulados", yscale="log")
plt.tight_layout(); plt.show()

Los ganadores no son los mejores ítems: son los que tuvieron suerte al principio. **Tu log de entrenamiento lleva la firma de tu política anterior.** Las respuestas de la industria: exploración explícita (bandits, módulo 14), corrección por propensión (IPS), datasets con exposición aleatoria (KuaiRand, Coat, Yahoo! R3) y re-ranking por diversidad (módulo 13).

---
## 5. Construir señales implícitas

En streaming casi no hay ratings; hay **reproducciones**. A partir de un log de *plays* puedes construir varias señales:

| Señal | Definición | Pros / contras |
|---|---|---|
| Play binario | $y_{ui} = 1$ si reprodujo | mucha señal, mucho ruido (autoplay, curiosidad) |
| Completado | $y_{ui} = 1$ si % visto $\geq \tau$ (p. ej. 70 %) | más fiable, más escasa; sesgo hacia contenido corto |
| Tiempo visto | minutos, $\log(1+\text{min})$ | favorece contenido largo (normalizar por duración) |
| Confianza (Hu et al., 2008) | $p_{ui} = \mathbb{1}[r_{ui}>0]$, $c_{ui} = 1 + \alpha \log(1 + r_{ui}/\epsilon)$ | separa **preferencia** de **confianza**; base de ALS implícito (módulo 05) |
| Negativo explícito | impresión sin play, "no me interesa", abandono < 5 % | oro puro si lo logueas |

MovieLens solo tiene ratings, así que **simulamos un log de reproducciones** coherente con ellos (con duraciones y % visto que dependen del rating) para practicar.

In [ ]:
def simular_log_plays(ratings: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    """Un play por rating, con % visto dependiente del rating, + impresiones sin play."""
    r = np.random.default_rng(seed)
    dur = pd.Series(r.integers(80, 180, ratings.item_id.max() + 1))  # minutos por película
    medias = {1: 0.15, 2: 0.35, 3: 0.65, 4: 0.85, 5: 0.95}
    m = ratings.rating.round().astype(int).map(medias).to_numpy()
    frac = np.clip(r.beta(m * 6, (1 - m) * 6), 0.01, 1.0)
    plays = ratings[["user_id", "item_id", "timestamp"]].assign(
        evento="play", frac_vista=frac, minutos=frac * dur.loc[ratings.item_id].to_numpy(),
        rating=ratings.rating.to_numpy())
    imp = ratings.sample(frac=0.5, random_state=seed)[["user_id", "timestamp"]].copy()
    imp["item_id"] = r.integers(1, ratings.item_id.max() + 1, len(imp))
    imp = imp.assign(evento="impresión", frac_vista=0.0, minutos=0.0, rating=np.nan)
    return pd.concat([plays, imp]).sort_values("timestamp").reset_index(drop=True)

eventos = simular_log_plays(ratings)
plays = eventos[eventos.evento == "play"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
plays.boxplot(column="frac_vista", by="rating", ax=axes[0], grid=False)
axes[0].set(title="% visto según el rating (log simulado)", xlabel="rating", ylabel="fracción vista"); fig.suptitle("")
alpha_ = 2.0
conf = 1 + alpha_ * np.log1p(plays.minutos / 10)
axes[1].hist(conf, bins=40, color=COL["verde"])
axes[1].set(title=r"Confianza $c_{ui} = 1 + \alpha\log(1 + \mathrm{min}/10)$", xlabel="confianza", ylabel="nº plays")
plt.tight_layout(); plt.show()
eventos.evento.value_counts()

In [ ]:
# ¿Qué umbral de "completado" se parece más a "le gustó" (rating >= 4)?
gusta = plays.rating >= 4
for tau in [0.05, 0.3, 0.5, 0.7, 0.9]:
    pos = plays.frac_vista >= tau
    prec = (pos & gusta).sum() / pos.sum()
    rec = (pos & gusta).sum() / gusta.sum()
    print(f"τ = {tau:.2f}: positivos = {pos.mean():5.1%}  precisión vs 'le gustó' = {prec:.2f}  recall = {rec:.2f}")

En el resto del curso usaremos la convención estándar en la literatura para MovieLens: **positivo implícito = rating ≥ 4** (`cd.to_implicit(ratings, threshold=4.0)`). Es la opción más común en papers (aunque otros usan "cualquier rating" como positivo: ¡comprueba siempre qué hace un paper antes de comparar números!).

---
## 6. Limpieza: duplicados, k-core y outliers

- **Duplicados** (re-visionados, doble clic): decide si agregas (contar, último timestamp) o eliminas. `cd.dedup_interactions`.
- **k-core**: quedarse solo con usuarios con $\geq k_u$ interacciones e ítems con $\geq k_i$. Es **iterativo**: quitar ítems raros puede dejar a un usuario por debajo del umbral, y viceversa.
- **Outliers**: cuentas compartidas, bots, usuarios con miles de eventos por día. En producción se filtran con reglas (eventos/min, sesiones imposibles).

In [ ]:
def k_core(df: pd.DataFrame, k_user: int, k_item: int) -> pd.DataFrame:
    while True:
        n0 = len(df)
        df = df[df.item_id.map(df.item_id.value_counts()) >= k_item]
        df = df[df.user_id.map(df.user_id.value_counts()) >= k_user]
        if len(df) == n0:
            return df

positivos = cd.to_implicit(ratings, 4.0)
filas_k = []
for k in [1, 2, 5, 10, 20, 30]:
    f = k_core(positivos, k, k)
    filas_k.append({"k": k, "interacciones": len(f), "usuarios": f.user_id.nunique(), "ítems": f.item_id.nunique(),
                    "densidad": len(f) / max(1, f.user_id.nunique() * f.item_id.nunique())})
tabla_k = pd.DataFrame(filas_k).set_index("k")
assert len(k_core(positivos, 5, 5)) == len(cd.filter_k_core(positivos, 5, 5))   # coincide con el módulo
fig, ax = plt.subplots(figsize=(7.5, 3.4))
(tabla_k[["interacciones", "usuarios", "ítems"]] / tabla_k.loc[1, ["interacciones", "usuarios", "ítems"]]).plot(
    ax=ax, marker="o", title="Lo que sobrevive al k-core (fracción respecto a k=1)")
ax.set(xlabel="k (mínimo por usuario y por ítem)", ylabel="fracción"); plt.tight_layout(); plt.show()
tabla_k

> 🧠 El k-core **sube la densidad y las métricas** (los ítems y usuarios difíciles desaparecen), por eso muchos papers usan 10-core o 20-core. Pero en producción **no puedes** quitar a los usuarios nuevos: un modelo evaluado en 20-core está evaluado sobre un mundo más fácil que el real. Y aplicarlo **antes** de partir en train/test usa información del futuro para decidir quién entra. En CineMatch aplicamos el k-core **solo al train**.

---
## 7. Splits y *data leakage*

Esta es la sección más importante del módulo. Hay tres familias de split:

| Split | Cómo | Usado en | Problema |
|---|---|---|---|
| **Aleatorio** | cada interacción va a test con prob. $p$ | muchos papers antiguos | mezcla futuro y pasado: entrenas con lo que pasó **después** de lo que predices |
| **Leave-one-out (LOO)** | la **última** interacción de cada usuario a test | NCF, SASRec, BERT4Rec… | el corte es distinto para cada usuario: el train de Ana incluye eventos posteriores al test de Bruno (**leakage entre usuarios**) |
| **Temporal global** | un instante $T$: train $= \{t < T\}$, test $= \{t \geq T\}$ | la industria; papers rigurosos | test con usuarios/ítems nuevos (cold start) → hay que decidir qué hacer con ellos |

> 💡 **Si vienes de series temporales o *forecasting***: el split temporal global es tu *walk-forward validation* (`TimeSeriesSplit`); hacer un split aleatorio en recomendación es como barajar un dataset de ventas antes de predecir el mes que viene. Y el *leakage* de la sección siguiente es el mismo *target leakage* que ya conoces de ML tabular, solo que más fácil de cometer: basta con un `value_counts()` antes del split.

El split temporal global es lo único que **reproduce lo que pasa en producción**: entrenas el lunes con todo lo anterior y sirves el martes. Ji, Sun, Zhang & Li (2023, *A Critical Study on Data Leakage in Recommender System Offline Evaluation*) mostraron que el leakage del LOO/aleatorio puede **cambiar qué modelo gana**.

In [ ]:
def dibujar_split(ax, df, es_test, titulo, n_users=12):
    us = df.user_id.drop_duplicates().sample(n_users, random_state=1)
    for k, u in enumerate(us):
        d = df[df.user_id == u]
        t = (d.timestamp - df.timestamp.min()) / 86_400
        m = es_test.loc[d.index]
        ax.scatter(t[~m], np.full((~m).sum(), k), s=9, color=COL["train"])
        ax.scatter(t[m], np.full(m.sum(), k), s=14, color=COL["test"])
    ax.set(title=titulo, xlabel="días desde el inicio", yticks=[])

pos = positivos.copy()
es_rand = pd.Series(rng.random(len(pos)) < 0.2, index=pos.index)
rank_fin = pos.sort_values("timestamp").groupby("user_id").cumcount(ascending=False)
es_loo = (rank_fin == 0).reindex(pos.index)
T = pos.timestamp.quantile(0.8)
es_temp = pos.timestamp >= T
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
dibujar_split(axes[0], pos, es_rand, "Aleatorio (20 %)")
dibujar_split(axes[1], pos, es_loo, "Leave-one-out (último por usuario)")
dibujar_split(axes[2], pos, es_temp, "Temporal global (último 20 % del tiempo)")
axes[2].axvline((T - pos.timestamp.min()) / 86_400, color="k", ls="--")
axes[0].set_ylabel("usuarios (muestra)")
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=COL["train"], label="train"),
                    plt.Line2D([], [], marker="o", ls="", color=COL["test"], label="test")], loc="upper right")
plt.tight_layout(); plt.show()

In [ ]:
# ¿Cuánto "futuro" hay en el train del leave-one-out?
tr_loo, te_loo = cd.leave_one_out_split(positivos)
t_test_mediano = te_loo.timestamp.median()
print(f"LOO: {np.mean(tr_loo.timestamp > t_test_mediano):.1%} de las interacciones de train son POSTERIORES "
      "al instante mediano de test")
print(f"LOO: {np.mean(tr_loo.timestamp > te_loo.timestamp.min()):.1%} de train es posterior al primer evento de test")
tr_t, va_t, te_t = cd.temporal_split(positivos, val_frac=0.1, test_frac=0.1)
print(f"Temporal: max(train) < min(val) ≤ min(test): {tr_t.timestamp.max() < va_t.timestamp.min() <= te_t.timestamp.min()}")
print(f"Temporal: tamaños train/val/test = {len(tr_t):,} / {len(va_t):,} / {len(te_t):,} · "
      f"usuarios en test: {te_t.user_id.nunique()}")

### 🧪 Experimento: el mismo baseline, cuatro "verdades"
Evaluamos **el mismo** recomendador de popularidad con distintos splits, y una versión con *leakage* descarado: calcular la popularidad con **todo** el dataset (incluido el test). Métrica: **Recall@10** (fracción de los ítems de test del usuario que aparecen en su top-10; la formalizamos en el módulo 02).

In [ ]:
def recall_at_k(recs: dict, test: pd.DataFrame, k: int = 10) -> float:
    rel = test.groupby("user_id").item_id.agg(set)
    return float(np.mean([len(r & set(recs.get(u, [])[:k])) / len(r) for u, r in rel.items()]))

def eval_popularidad(train, test, fuente_popularidad=None, k=10):
    fuente = train if fuente_popularidad is None else fuente_popularidad
    ranking = fuente.item_id.value_counts().index.to_numpy()
    vistos = train.groupby("user_id").item_id.agg(set)
    recs = {u: [i for i in ranking[: k + len(vistos.get(u, ()))] if i not in vistos.get(u, set())][:k]
            for u in test.user_id.unique()}
    return recall_at_k(recs, test, k)

tr_r, te_r = cd.random_split(positivos, 0.2)
tr_l, te_l = cd.leave_one_out_split(positivos)
tr_g, _, te_g = cd.temporal_split(positivos, val_frac=0.0, test_frac=0.2)
res_split = {
    "Aleatorio": eval_popularidad(tr_r, te_r),
    "Leave-one-out": eval_popularidad(tr_l, te_l),
    "Temporal global": eval_popularidad(tr_g, te_g),
    "Temporal + LEAKAGE\n(popularidad con test)": eval_popularidad(tr_g, te_g, fuente_popularidad=positivos),
}
fig, ax = plt.subplots(figsize=(8.5, 3.4))
ax.bar(list(res_split), list(res_split.values()), color=[COL["gris"], COL["val"], COL["train"], COL["test"]])
ax.set(ylabel="Recall@10", title="Popularidad: mismo modelo, distinta evaluación")
for k, v in enumerate(res_split.values()):
    ax.text(k, v, f"{v:.3f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

Cada split mide **una tarea distinta** (y con usuarios distintos), así que los números no son comparables entre sí — esa es justo la lección: **un número de evaluación no significa nada sin su protocolo**. Lo que sí es inequívoco es la comparación entre las dos últimas barras (mismo split, mismos usuarios): usar el test para calcular cualquier cosa del modelo (popularidad, normalizaciones, vocabularios, k-core, estadísticas de *features*) le regala información del futuro. Aquí, con un modelo de una sola "feature", el efecto es pequeño; con cientos de *features* agregadas (p. ej. "nº de vistas totales del ítem" calculada sobre la tabla completa, o el CTR histórico del ítem) es exactamente así como un modelo mediocre parece excelente offline y se hunde online.

---
## 8. Baselines fuertes

Antes de cualquier modelo, **siempre**:

1. **Aleatorio** — el suelo; si no lo bates por mucho, hay un bug.
2. **Popularidad global** — sorprendentemente difícil de batir en un split temporal.
3. **Popularidad reciente** — popularidad en una ventana de $W$ días, o con decaimiento exponencial $w(e) = 0{,}5^{\,\text{edad}(e)/h}$ con vida media $h$. En catálogos con estrenos (streaming, noticias, moda) suele ganar a la global.
4. **Popularidad por segmento** — por país, edad, dispositivo, género favorito… el primer paso de personalización.

Los hiperparámetros ($W$, $h$) se ajustan en **validación** y se reportan en **test**, como cualquier modelo.

In [ ]:
train, val, test = cd.temporal_split(positivos, val_frac=0.1, test_frac=0.1)
ventanas = [3, 7, 14, 30, 60, 120, None]
res_w = {}
for w in ventanas:
    recs = cd.recommend_popular(train, val.user_id.unique(), k=10, window_days=w)
    res_w["todo" if w is None else f"{w} d"] = recall_at_k(recs, val, 10)
mejor_w = max(res_w, key=res_w.get)
fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(list(res_w), list(res_w.values()), marker="o", color=COL["train"])
ax.set(xlabel="ventana de popularidad", ylabel="Recall@10 (validación)",
       title=f"Ajuste de la ventana en validación (mejor: {mejor_w})")
plt.tight_layout(); plt.show()

In [ ]:
W_BEST = None if mejor_w == "todo" else int(mejor_w.split()[0])
train_full = pd.concat([train, val])          # para test: reentrenamos con train + val
usuarios_test = test.user_id.unique()

if users is not None:   # segmento = grupo de edad (MovieLens trae demografía)
    seg = pd.cut(users.set_index("user_id").age, [0, 18, 25, 35, 45, 120], labels=False).to_dict()
else:                   # sin demografía: segmento = nivel de actividad del usuario
    seg = pd.qcut(train_full.user_id.value_counts(), 4, labels=False).to_dict()

baselines = {
    "Aleatorio": cd.recommend_random(train_full, usuarios_test, k=10),
    "Popularidad global": cd.recommend_popular(train_full, usuarios_test, k=10),
    f"Popularidad reciente ({mejor_w})": cd.recommend_popular(train_full, usuarios_test, k=10, window_days=W_BEST),
    "Popularidad decaída (h=30 d)": cd.recommend_popular(train_full, usuarios_test, k=10, half_life_days=30),
    "Popularidad por segmento": cd.recommend_popular_by_segment(train_full, usuarios_test, seg, k=10),
}
res_b = pd.Series({n: recall_at_k(r, test, 10) for n, r in baselines.items()}).sort_values()
ax = res_b.plot.barh(figsize=(8, 3.2), color=COL["train"], title="Baselines en test (split temporal global)")
ax.set_xlabel("Recall@10"); plt.tight_layout(); plt.show()
res_b.round(4)

> 🧪 Con datos reales de MovieLens, la popularidad (global o reciente) multiplica por un orden de magnitud al aleatorio. Este es el listón para **todo** lo que construyas a partir del módulo 03.

Todas estas funciones viven en `cinematch_data.py`. El pipeline completo en una sola llamada:
```python
data = cd.prepare_cinematch("1m")     # dict con train / val / test / items / ratings
```

---
## 9. 📚 Catálogo de datasets del mundo real

| Dataset | Dominio | Escala aprox. | Feedback | Qué lo hace especial | Dónde |
|---|---|---|---|---|---|
| **MovieLens** 100K/1M/25M/32M | películas | 100K – 32M ratings | explícito 1–5 + timestamps (+ tags, géneros) | el estándar; demografía en 100K/1M | <https://grouplens.org/datasets/movielens/> |
| **Netflix Prize** | películas | 100M ratings, 480K usuarios | explícito 1–5 | histórico (2006); retirado oficialmente, circula en Kaggle | <https://www.kaggle.com/datasets/netflix-inc/netflix-prize-data> |
| **Amazon Reviews 2023** | e-commerce | ~571M reseñas, 33 categorías | ratings + texto + metadatos | texto e imágenes ricos; splits oficiales | <https://amazon-reviews-2023.github.io/> |
| **Yelp Open Dataset** | negocios locales | ~7M reseñas | ratings + texto + check-ins | geografía, grafos sociales | <https://www.yelp.com/dataset> |
| **MIND** | noticias | 1M usuarios, 15M impresiones | clics **con impresiones** | negativos reales; cold start extremo | <https://msnews.github.io/> |
| **KuaiRec** | vídeo corto | 1.411 usuarios × 3.327 vídeos **totalmente observada** (submatriz) | watch ratio | permite evaluación sin sesgo de exposición | <https://kuairec.com/> |
| **KuaiRand** | vídeo corto | 12 señales, millones de eventos | multi-señal + **exposición aleatoria** | debiasing, multi-tarea | <https://kuairand.com/> |
| **RetailRocket** | e-commerce | 2,7M eventos | view / add-to-cart / transaction | sesiones, implícito multinivel | <https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset> |
| **Yoochoose** (RecSys Challenge 2015) | e-commerce | 33M clics | clics + compras por sesión | clásico de recomendación por sesión (GRU4Rec) | <https://recsys.acm.org/recsys15/challenge/> |
| **H&M Personalized Fashion** | moda | 31M transacciones | compras + imágenes + metadatos | multimodal, estacionalidad | <https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations> |
| **OTTO** | e-commerce | 12M sesiones, 220M eventos | clicks / carts / orders | multi-objetivo, sesiones largas | <https://github.com/otto-de/recsys-dataset> |
| **Last.fm 1K / 360K** | música | 19M scrobbles (1K) | escuchas implícitas | secuencias de escucha | <http://ocelma.net/MusicRecommendationDataset/> |
| **LFM-2b** | música | 2.000M escuchas | escuchas + demografía | fairness; **retirado** por licencias (CHIIR 2022) | <https://www.cp.jku.at/datasets/LFM-2b/> |
| **Goodbooks-10k** | libros | 6M ratings | explícito 1–5 + "to read" | alternativo a MovieLens | <https://github.com/zygmuntz/goodbooks-10k> |
| **Criteo** (Kaggle DAC / 1TB) | publicidad (CTR) | 45M – 4.000M impresiones | clic sí/no, 13 num. + 26 cat. | estándar de modelos CTR (módulo 06) | <https://ailab.criteo.com/ressources/> |
| **Avazu** | publicidad (CTR) | 40M impresiones | clic sí/no | CTR móvil (módulo 06) | <https://www.kaggle.com/c/avazu-ctr-prediction> |
| **Open Bandit Dataset** (ZOZOTOWN) | moda | 26M impresiones | clic + **propensiones reales** | off-policy evaluation (módulo 14) | <https://research.zozo.com/data.html> |

> ⚠️ Revisa siempre la **licencia** (casi todos son solo para investigación, no comercial) y si el dataset trae **impresiones** (negativos reales) o solo positivos.

---
## 10. 🧪 Negativos: lo que no viste no es lo que no te gusta

Con feedback implícito solo tienes positivos. Para entrenar (BPR, two-tower, BCE) necesitas **negativos**, y casi siempre se **muestrean** de lo no observado. Pero algunos de esos "negativos" son ítems que el usuario **verá y le gustarán en el futuro** (*falsos negativos*). Midámoslo: muestreamos negativos para los usuarios de train y miramos cuántos aparecen en su test.

In [ ]:
def tasa_falsos_negativos(train, test, n_neg=100, modo="uniforme", seed=SEED):
    r = np.random.default_rng(seed)
    catalogo = train.item_id.unique()
    p = None
    if modo == "popularidad":
        cnt = train.item_id.value_counts().reindex(catalogo).to_numpy(float)
        p = cnt ** 0.75 / (cnt ** 0.75).sum()          # como word2vec
    vistos = train.groupby("user_id").item_id.agg(set)
    futuros = test.groupby("user_id").item_id.agg(set)
    fn, total = 0, 0
    for u, fut in futuros.items():
        neg = r.choice(catalogo, size=n_neg, p=p)
        neg = [i for i in neg if i not in vistos.get(u, set())]
        fn += sum(i in fut for i in neg); total += len(neg)
    return fn / total

tasas = {m: tasa_falsos_negativos(tr_t, te_t, modo=m) for m in ["uniforme", "popularidad"]}
ratio_base = te_t.groupby("user_id").size().mean() / tr_t.item_id.nunique()
print({k: f"{v:.3%}" for k, v in tasas.items()}, f"· (referencia: |test_u| / |I| = {ratio_base:.3%})")
pd.Series(tasas).plot.bar(rot=0, color=[COL["train"], COL["val"]], figsize=(6, 3),
                          title="% de negativos muestreados que el usuario consumirá después")
plt.ylabel("tasa de falsos negativos"); plt.tight_layout(); plt.show()

El muestreo por popularidad da negativos **más difíciles** (más informativos) pero con **más falsos negativos**. Es el *trade-off* que reaparecerá en el two-tower (módulo 08: *hard negatives*, corrección logQ) y en las *sampled metrics* (módulo 02).

---
## 11. 🏭 En producción

- **El log de impresiones es el activo más valioso.** YouTube, Netflix o Meta registran qué se mostró, en qué posición y en qué contexto, no solo lo que se clicó. Sin impresiones no hay negativos fiables, ni corrección de sesgo de posición, ni *off-policy evaluation*.
- **Ventanas de entrenamiento móviles**: en lugar de "todo el histórico", entrenar con los últimos N días, con más peso a lo reciente. Monolith (ByteDance, 2022) lleva esto al extremo con entrenamiento en tiempo real.
- **Point-in-time correctness**: las *features* de cada ejemplo de entrenamiento deben calcularse **con lo que se sabía en ese instante**. Los *feature stores* (Feast, Tecton) existen en buena parte para garantizar esto y evitar el leakage del apartado 7 (módulo 16).
- **Validación de datos en el pipeline**: esquemas y rangos (Pandera / Great Expectations), volumen de eventos por hora, proporción de eventos por tipo… un cambio de logging en el cliente puede romper silenciosamente el modelo (módulo 17).
- **Señales compuestas**: los objetivos de ranking reales son mezclas (clic, play, % visto, like, compartir, retención) — ver MMoE de YouTube (Zhao et al., 2019) en el módulo 07.

## 🧠 Secretos de la élite

1. **El split temporal global es innegociable** para decidir qué modelo va a producción. Ji et al. (2023) muestran que el leakage de los splits aleatorios/LOO cambia el ranking de modelos. Si un paper usa LOO, sus números no son comparables con tu evaluación temporal.
2. **El split forma parte del resultado.** Meng, McCreadie, Macdonald & Ounis (RecSys 2020, *Exploring Data Splitting Strategies for the Evaluation of Recommendation Models*) evaluaron los mismos modelos con distintos splits y el **orden relativo de los algoritmos cambiaba**. Por eso la popularidad *reciente* (ventana ajustada en validación) tiene que estar siempre en tu tabla: en catálogos con estrenos es el baseline temporal que más modelos dejan en evidencia.
3. **Toda estadística del pipeline se calcula solo con train**: popularidad, vocabularios de IDs, medias para normalizar, k-core, umbrales. Un `value_counts()` sobre el DataFrame completo es leakage.
4. **El k-core infla las métricas.** Reporta siempre qué filtrado usas; evalúa también en los usuarios con poco historial, que en producción son mayoría.
5. **"Positivo = rating ≥ 4" vs "positivo = cualquier rating"** pueden duplicar o dividir las métricas: muchas discrepancias entre papers vienen solo de aquí.
6. **Los falsos negativos existen**: con muestreo por popularidad, los negativos difíciles a veces son simplemente positivos futuros. Las correcciones (logQ, eliminar negativos accidentales) importan.
7. **Loguea impresiones desde el día 1.** Es imposible reconstruirlas después y son la base de la corrección de sesgos, la exploración y la evaluación off-policy.
8. **Cuenta los usuarios de tu test antes de mirar la métrica.** En ML-100K el último 10 % del tiempo solo tiene ~50 usuarios (casi todos valoran en una ráfaga al registrarse): ahí la popularidad de 30 días «gana» por 4× a la global por puro ruido. Por eso CineMatch usa ML-1M (~950 usuarios en test) y en el módulo 02 aprenderás a ponerle intervalos de confianza a todo.

## ⚠️ Errores comunes

- Calcular la popularidad (o cualquier *feature*) sobre el dataset completo antes del split.
- Hacer k-core sobre todo el dataset y luego partir.
- Partir aleatoriamente "porque es lo que hace sklearn".
- Olvidar excluir los ítems ya vistos al recomendar (o, al revés, excluirlos cuando el producto permite re-consumo: música, compras recurrentes).
- No tratar a los usuarios/ítems de test que no existen en train (decide: los excluyes, o los sirves con popularidad, y **dilo**).
- Tratar `NaN` de la matriz como 0 en algoritmos explícitos (asumes que todo lo no visto es un 0 estrellas).
- Comparar números de papers con distinto dataset, filtrado, umbral de positivo y split.

---
## 📝 Autoevaluación

**1.** ¿Por qué CSR y no CSC para obtener el historial de un usuario?
<details><summary>Respuesta</summary>CSR guarda las filas (usuarios) contiguas: el historial de u son indices[indptr[u]:indptr[u+1]], acceso O(longitud del historial). CSC es lo mismo por columnas (ítems).</details>

**2.** Un modelo da Recall@10 = 0,35 con split aleatorio y 0,12 con split temporal. ¿Cuál reportas y por qué?
<details><summary>Respuesta</summary>El temporal: reproduce la situación de producción (entrenar con el pasado, predecir el futuro). El aleatorio deja que el modelo use interacciones posteriores a las que predice y puede incluso cambiar qué modelo gana (Ji et al.).</details>

**3.** ¿Qué es MNAR y qué consecuencia tiene para un modelo que predice ratings?
<details><summary>Respuesta</summary>Missing Not At Random: la probabilidad de que un rating se observe depende de su valor (se valora lo que gusta). El modelo entrenado solo en observados sobreestima los ratings de lo no visto y su RMSE en observados no refleja el error real sobre todo el catálogo.</details>

**4.** ¿Por qué el leave-one-out tiene leakage aunque el test de cada usuario sea su última interacción?
<details><summary>Respuesta</summary>Porque el corte es distinto para cada usuario: el train contiene interacciones de otros usuarios posteriores al instante de test, así que el modelo "conoce" tendencias del futuro (p. ej. una película que se hizo popular después).</details>

**5.** ¿Cómo ajustas la ventana de la popularidad reciente sin hacer trampa?
<details><summary>Respuesta</summary>Split temporal en train/val/test; eliges la ventana que maximiza la métrica en val usando solo train; luego recalculas con train+val y reportas una única vez en test.</details>

**6.** Nombra dos datasets con negativos reales (impresiones) y para qué sirven.
<details><summary>Respuesta</summary>MIND (noticias, impresiones con clic/no clic), KuaiRand (exposición aleatoria) y Open Bandit Dataset (con propensiones). Sirven para entrenar con negativos reales, estudiar sesgos y evaluar off-policy.</details>

**7.** ¿Qué *trade-off* hay entre muestrear negativos uniformemente o por popularidad?
<details><summary>Respuesta</summary>Los negativos populares son más difíciles e informativos (el modelo aprende a no recomendar solo lo popular), pero tienen más probabilidad de ser falsos negativos y sesgan las puntuaciones (requieren corrección, p. ej. logQ).</details>

**8. (Cálculo)** CineMatch tiene 2 M de usuarios, 50.000 títulos y 300 M de interacciones. ¿Cuánto ocupa la matriz en denso (float32) y en CSR (float32 + índices int32)?
<details><summary>Respuesta</summary>Denso: 2·10⁶ × 5·10⁴ × 4 B = 4·10¹¹ B = 400 GB. CSR ≈ 8 × 3·10⁸ + 4 × (2·10⁶ + 1) ≈ 2,4 GB. Densidad = 3·10⁸ / 10¹¹ = 0,3 %. Por eso todo el curso trabaja con <code>scipy.sparse</code>.</details>

**9. (Diagnóstico)** Un compañero añade la *feature* «nº total de vistas del ítem» al modelo y el Recall@10 offline salta de 0,08 a 0,15. En el A/B no mejora nada. ¿Cuál es tu primera hipótesis y cómo la compruebas?
<details><summary>Respuesta</summary>Leakage temporal: la <i>feature</i> se calculó sobre la tabla completa (incluido el periodo de test), así que codifica qué se verá en el futuro. Se comprueba recalculándola solo con eventos anteriores al corte (o <i>point-in-time</i> por ejemplo, módulo 16) y viendo si la ganancia offline desaparece.</details>

---
## 📚 Referencias

- Harper & Konstan (2015). *The MovieLens Datasets: History and Context*. ACM TiiS. <https://doi.org/10.1145/2827872>
- Hu, Koren & Volinsky (2008). *Collaborative Filtering for Implicit Feedback Datasets*. ICDM. <https://doi.org/10.1109/ICDM.2008.22>
- Marlin & Zemel (2009). *Collaborative Prediction and Ranking with Non-Random Missing Data*. RecSys. <https://doi.org/10.1145/1639714.1639717>
- Joachims, Swaminathan & Schnabel (2017). *Unbiased Learning-to-Rank with Biased Feedback*. WSDM. <https://arxiv.org/abs/1608.04468>
- Chaney, Stewart & Engelhardt (2018). *How Algorithmic Confounding in Recommendation Systems Increases Homogeneity and Decreases Utility*. RecSys. <https://arxiv.org/abs/1710.11214>
- Chen et al. (2023). *Bias and Debias in Recommender System: A Survey and Future Directions*. ACM TOIS. <https://arxiv.org/abs/2010.03240>
- Ji, Sun, Zhang & Li (2023). *A Critical Study on Data Leakage in Recommender System Offline Evaluation*. ACM TOIS. <https://arxiv.org/abs/2010.11060>
- Meng, McCreadie, Macdonald & Ounis (2020). *Exploring Data Splitting Strategies for the Evaluation of Recommendation Models*. RecSys. <https://arxiv.org/abs/2007.13237>
- Sun et al. (2020). *Are We Evaluating Rigorously? Benchmarking Recommendation for Reproducible Evaluation and Fair Comparison*. RecSys. <https://doi.org/10.1145/3383313.3412489>
- Wu et al. (2020). *MIND: A Large-scale Dataset for News Recommendation*. ACL. <https://aclanthology.org/2020.acl-main.331/>
- Gao et al. (2022). *KuaiRec: A Fully-observed Dataset and Insights for Evaluating Recommender Systems*. CIKM. <https://arxiv.org/abs/2202.10842>
- Gao et al. (2022). *KuaiRand: An Unbiased Sequential Recommendation Dataset with Randomly Exposed Videos*. CIKM. <https://arxiv.org/abs/2208.08696>
- Hou et al. (2024). *Bridging Language and Items for Retrieval and Recommendation* (Amazon Reviews 2023). <https://arxiv.org/abs/2403.03952>
- Saito, Aihara, Matsutani & Narita (2021). *Open Bandit Dataset and Pipeline: Towards Realistic and Reproducible Off-Policy Evaluation*. NeurIPS Datasets & Benchmarks. <https://arxiv.org/abs/2008.07146>
- Schedl et al. (2022). *LFM-2b: A Dataset of Enriched Music Listening Events for Recommender Systems Research and Fairness Analysis*. CHIIR. <https://doi.org/10.1145/3498366.3505791>
- Liu et al. (2022). *Monolith: Real Time Recommendation System With Collisionless Embedding Table*. <https://arxiv.org/abs/2209.07663>
- SciPy sparse: <https://docs.scipy.org/doc/scipy/reference/sparse.html>